# 1. Demand

## 1.0 Setup

*(code: import thư viện, đọc dữ liệu)*

### 1.0.1 Input data

**(1) `demand_b2b` — cầu B2B năm 2025 của công ty** (Decision_Log 5.1)
Mỗi dòng = 1 ship-to × 1 sản phẩm. Đơn vị **cái/năm** (hàng đã giao tháng 1–11/2025, công ty quy ra cả năm). 5,733,456 dòng, 275.3 triệu cái, 3,347 nhóm khách, 1,735 sản phẩm. Đã lọc ở `Validation.ipynb` 2.13e: bỏ Malaysia, bỏ sản phẩm lỗi (5.4).

| Trường | Kiểu | Ý nghĩa | Đơn vị | Dùng để |
|---|---|---|---|---|
| `ShipToGroupID` | mã | Nhóm khách = **nút cầu k** của mô hình | — | Gom cầu theo nhóm |
| `ShipToID` | mã | Điểm giao hàng | — | Đếm số điểm giao mỗi nhóm |
| `ProductGroupID` | mã | Nhóm sản phẩm của công ty | — | Không dùng (mô hình gộp 1 sản phẩm, 2.3) |
| `ProductID` | mã | Mã sản phẩm | — | Tra cái/pallet, trọng lượng |
| `DemandInPCS` | số | Cầu | cái/năm | Tính cầu |

**(2) `product_master` — danh mục sản phẩm**

| Trường | Kiểu | Ý nghĩa | Đơn vị | Dùng để |
|---|---|---|---|---|
| `ProductID` | mã | Khoá nối với `demand_b2b` | — | Nối bảng |
| `QuantityPiecesPerPallet` | số | 1 pallet chứa bao nhiêu cái | cái/pallet | Đổi cái → pallet |
| `GrossWeightInGram` | số | Trọng lượng 1 cái (cả bao bì) | gram | Đổi cái → tấn (cho CO2, `f3`) |

**(3) `shipto_group_map`** — 34,430 điểm giao → 3,347 nhóm (Decision_Log 4.6). Dùng để lấy danh sách nhóm khách.

**(4) `customer_master`** — đảo của từng điểm giao (`MainIsland`). Dùng để gắn đảo cho mỗi nhóm khách.

In [1]:
# ===== 1.0 SETUP =====
import pandas as pd                       # xử lý bảng
import numpy as np                        # tính toán
from pathlib import Path                  # ghép đường dẫn

pd.set_option("display.width", 200)       # in bảng rộng, không bị xuống dòng

IN = Path("../Model - Data used")         # data đã clean ở 2 notebook Validation (chỉ đọc)
OUT = Path("../Model Outputs")            # nơi lưu kết quả của các notebook model
OUT.mkdir(exist_ok=True)                  # tạo folder nếu chưa có

# Bảng gán khách -> nhóm: ShipToID (điểm giao hàng) -> ShipToGroupID (nhóm khách = nút cầu k của mô hình)
mp = pd.read_parquet(IN / "shipto_group_map.parquet")
G = sorted(mp["ShipToGroupID"].unique())                                 # tập nhóm khách, sắp xếp để thứ tự cố định
print(f"Ship-to: {len(mp):,} | nhóm khách: {len(G):,}")                  # kỳ vọng 34.430 ship-to, 3.347 nhóm

Ship-to: 34,430 | nhóm khách: 3,347


## 1.1 Theoretical background

### 1.1.1 Demand uncertainty in distribution network design

*(nội dung: vì sao cầu bất định; các cách xử lý: tất định / ngẫu nhiên (stochastic) / robust; luận văn dùng cầu tất định `d_k` và kiểm tra bằng phân tích độ nhạy +10%, +20% — Decision_Log 1.3, 6.1)*

## 1.2 Demand data construction

### 1.2.1 From pieces to pallets

Mô hình tính phí chở, phí kho và sức chứa theo **pallet** (Decision_Log 2.3). Mỗi sản phẩm chiếm chỗ khác nhau, nên phải đổi cái → pallet **theo từng sản phẩm trước**, rồi mới cộng lên (giống đổi từng loại ngoại tệ ra tiền Việt rồi mới cộng):

$$q^{pallet}_{\ell} = \frac{q^{pcs}_{\ell}}{\text{PcsPerPallet}_{p(\ell)}} \qquad q^{tấn}_{\ell} = \frac{q^{pcs}_{\ell} \times \text{GrossWeightInGram}_{p(\ell)}}{10^6}$$

- $\ell$: một dòng trong `demand_b2b`; $p(\ell)$: sản phẩm của dòng đó
- $q^{pcs}_{\ell}$: cầu (`DemandInPCS`, cái/năm)

Ví dụ: 8,000 thỏi son (4,000 cái/pallet) = 2 pallet; 3,000 chai dầu gội (1,000 cái/pallet) = 3 pallet → tổng 5 pallet. Cộng 11,000 cái rồi chia một con số chung sẽ sai.

Tấn tính kèm để dùng cho CO2 (`f3`, tính theo tấn-km).

In [2]:
# ===== 1.2.1 CÁI -> PALLET VÀ TẤN THEO TỪNG SẢN PHẨM =====
# Cầu công ty: 1 dòng = 1 ship-to × 1 sản phẩm, cái/năm (xem bảng 1.0.1)
dm = pd.read_parquet(IN / "demand_b2b.parquet")
print(f"Dòng: {len(dm):,} | cái/năm: {dm['DemandInPCS'].sum():,.0f}")                 # kỳ vọng 5.733.456 dòng, 275.299.863 cái

# Bảng tra sản phẩm: cái/pallet và trọng lượng 1 cái
pm = pd.read_parquet(IN / "product_master.parquet",
                     columns=["ProductID", "QuantityPiecesPerPallet", "GrossWeightInGram"])
pm["ProductID"] = pm["ProductID"].astype(str)                                        # cùng kiểu chuỗi với demand_b2b

# Gắn thông tin sản phẩm vào từng dòng cầu (validate: mỗi sản phẩm chỉ có 1 dòng trong master)
dm = dm.merge(pm, on="ProductID", how="left", validate="many_to_one")

# Kiểm tra: mọi dòng đều có cái/pallet và trọng lượng hợp lệ, nếu không thì dừng
assert dm["QuantityPiecesPerPallet"].notna().all() and (dm["QuantityPiecesPerPallet"] > 0).all(), "Thiếu / sai cái/pallet"
assert dm["GrossWeightInGram"].notna().all() and (dm["GrossWeightInGram"] > 0).all(), "Thiếu / sai trọng lượng"

# Mỗi dòng: pallet = cái ÷ cái/pallet của chính sản phẩm đó; tấn = cái × gram ÷ 1.000.000
dm["Pallet"] = dm["DemandInPCS"] / dm["QuantityPiecesPerPallet"]
dm["Tan"] = dm["DemandInPCS"] * dm["GrossWeightInGram"] / 1e6

print(f"Tổng pallet/năm: {dm['Pallet'].sum():,.0f} | tổng tấn/năm: {dm['Tan'].sum():,.0f} "
      f"| kg bình quân / pallet: {dm['Tan'].sum() * 1000 / dm['Pallet'].sum():,.0f}")    # kỳ vọng ~79.836 pallet, ~22.077 tấn

Dòng: 5,733,436 | cái/năm: 271,860,048


Tổng pallet/năm: 72,624 | tổng tấn/năm: 18,721 | kg bình quân / pallet: 258


### 1.2.2 Aggregation to demand groups

Mô hình chỉ cần cầu theo **nhóm khách** (nút cầu $k$), không cần từng ship-to hay từng sản phẩm. Cộng pallet và tấn của mọi dòng thuộc cùng một nhóm. Gắn thêm đảo của nhóm (mỗi nhóm nằm trọn trên 1 đảo, Decision_Log 4.6) để mô tả và để chọn `L_max` sau này.

In [3]:
# ===== 1.2.2 GOM THEO NHÓM KHÁCH =====
nhom = dm.groupby("ShipToGroupID").agg(
    Pallet=("Pallet", "sum"),                 # pallet/năm của nhóm
    Tan=("Tan", "sum"),                       # tấn/năm của nhóm
    SoShipTo=("ShipToID", "nunique"),         # số điểm giao trong nhóm
    SoSP=("ProductID", "nunique"),            # số sản phẩm khác nhau nhóm mua
)

# Đảo của nhóm: lấy từ đảo của các ship-to trong nhóm
cm = pd.read_parquet(IN / "customer_master.parquet", columns=["LocationID", "MainIsland"]).astype(str)
dao = (mp.astype(str).merge(cm, left_on="ShipToID", right_on="LocationID", how="left")
         .groupby("ShipToGroupID")["MainIsland"])
assert (dao.nunique() == 1).all(), "Có nhóm nằm trên >1 đảo"                       # Decision_Log 4.6
nhom["MainIsland"] = dao.first()

# Kiểm tra: nhóm nào cũng có cầu > 0. Số nhóm = 3.347 trừ nhóm chỉ gồm khách kênh Manufacturer (Decision_Log 5.10)
nhom = nhom.sort_index()                                                            # thứ tự cố định
assert nhom["Pallet"].notna().all() and (nhom["Pallet"] > 0).all(), "Có nhóm không có cầu"
assert set(nhom.index) <= set(mp["ShipToGroupID"].astype(str))                      # mọi nhóm đều có trong bảng gán
print(f"Nhóm khách: {len(nhom):,} | pallet/năm: {nhom['Pallet'].sum():,.0f} | tấn/năm: {nhom['Tan'].sum():,.0f}")

Nhóm khách: 3,346 | pallet/năm: 72,624 | tấn/năm: 18,721


## 1.3 Demand parameters

### 1.3.1 Demand per group d_k

$d_k$ = cầu của nhóm khách $k$, **pallet/năm** (mô hình 1 kỳ = 1 năm, Decision_Log 2.4). Đây là con số duy nhất về cầu đi vào mô hình (cầu tất định, 1.3, 6.1).

Kèm theo `TonPerPallet` = tấn ÷ pallet của từng nhóm: mỗi nhóm mua hỗn hợp sản phẩm khác nhau nên 1 pallet nặng khác nhau. Dùng để đổi pallet → tấn khi tính CO2 (`f3`).

In [4]:
# ===== 1.3.1 BẢNG CẦU CHO MÔ HÌNH =====
demand = pd.DataFrame({
    "ShipToGroupID": nhom.index,                             # nút cầu k
    "MainIsland": nhom["MainIsland"].values,                 # đảo của nhóm
    "DemandPallet": nhom["Pallet"].values,                   # cầu d_k, pallet/năm  -> vào mô hình
    "Ton": nhom["Tan"].values,                               # cầu, tấn/năm
    "TonPerPallet": (nhom["Tan"] / nhom["Pallet"]).values,   # tấn trên 1 pallet của nhóm (cho f3)
    "SoShipTo": nhom["SoShipTo"].values,                     # số điểm giao (mô tả)
    "SoSP": nhom["SoSP"].values,                             # số sản phẩm (mô tả)
})
print(demand.head().to_string(index=False))

ShipToGroupID MainIsland  DemandPallet        Ton  TonPerPallet  SoShipTo  SoSP
   105695-002       Java    654.616623 165.558200      0.252909         1  1525
   118438-598       Java   1218.473180 332.218737      0.272652         1   260
   124624-000       Java    687.256196 176.041349      0.256151         1   680
   128112-002       Java    414.874431 104.759743      0.252510         1  1310
   138789-000       Java    621.505794 149.609848      0.240722         1  1247


### 1.3.2 Descriptive statistics

Phân bố cầu theo nhóm, theo đảo, mức tập trung (bao nhiêu % nhóm lớn nhất chiếm bao nhiêu % cầu), và tấn/pallet.

In [5]:
# ===== 1.3.2 THỐNG KÊ MÔ TẢ =====
# (a) Phân bố cầu mỗi nhóm (pallet/năm): lệch phải -> vài nhóm rất lớn, đa số nhỏ
print("(a) DemandPallet mỗi nhóm (pallet/năm):")
print(demand["DemandPallet"].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())

# (b) Theo đảo: số nhóm, pallet/năm, tỷ trọng
theo_dao = demand.groupby("MainIsland").agg(so_nhom=("DemandPallet", "size"), pallet_nam=("DemandPallet", "sum"), tan_nam=("Ton", "sum"))
theo_dao["ty_trong"] = theo_dao["pallet_nam"] / theo_dao["pallet_nam"].sum()
print("\n(b) Theo đảo:")
print(theo_dao.sort_values("pallet_nam", ascending=False).round(3).to_string())

# (c) Mức tập trung: 1% / 10% / 20% nhóm lớn nhất chiếm bao nhiêu % cầu
s = demand["DemandPallet"].sort_values(ascending=False)
for q in [.01, .1, .2]:
    n = int(len(s) * q)
    print(f"(c) {q:.0%} nhóm lớn nhất ({n:,} nhóm) chiếm {s.head(n).sum() / s.sum():.1%} cầu")

# (d) Tấn / pallet theo nhóm: hỗn hợp sản phẩm khác nhau -> pallet nặng nhẹ khác nhau
print("\n(d) Tấn / pallet mỗi nhóm:")
print(demand["TonPerPallet"].describe(percentiles=[.05, .5, .95]).round(3).to_string())

(a) DemandPallet mỗi nhóm (pallet/năm):
count    3346.0
mean       21.7
std        68.0
min         0.0
25%         0.8
50%         3.0
75%        12.2
90%        48.0
99%       278.2
max      1251.2

(b) Theo đảo:
              so_nhom  pallet_nam    tan_nam  ty_trong
MainIsland                                            
Java             1762   43464.742  11228.230     0.598
Sumatra           798   13858.709   3584.003     0.191
Sulawesi          351    5558.281   1402.841     0.077
Kalimantan        255    5478.282   1409.665     0.075
Bali - Nusra      118    3459.312    893.228     0.048
Maluku             40     539.465    136.250     0.007
Papua              22     265.454     66.970     0.004
(c) 1% nhóm lớn nhất (33 nhóm) chiếm 23.8% cầu
(c) 10% nhóm lớn nhất (334 nhóm) chiếm 72.8% cầu
(c) 20% nhóm lớn nhất (669 nhóm) chiếm 86.9% cầu

(d) Tấn / pallet mỗi nhóm:
count    3346.000
mean        0.261
std         0.015
min         0.135
5%          0.240
50%         0.260
95%      

In [6]:
# ===== XEM NHÓM LỚN NHẤT VÀ NHỎ NHẤT =====
cot = ["ShipToGroupID", "MainIsland", "DemandPallet", "Ton", "SoShipTo", "SoSP"]
print("5 nhóm lớn nhất:")
print(demand.nlargest(5, "DemandPallet")[cot].round(2).to_string(index=False))
print("\n5 nhóm nhỏ nhất:")
print(demand.nsmallest(5, "DemandPallet")[cot].round(4).to_string(index=False))
# Bao nhiêu nhóm dưới 1 pallet/năm, và tổng của chúng chiếm bao nhiêu % cầu
nho = demand["DemandPallet"] < 1
print(f"\nNhóm < 1 pallet/năm: {nho.sum():,} nhóm | chiếm {demand.loc[nho, 'DemandPallet'].sum() / demand['DemandPallet'].sum():.2%} cầu")


5 nhóm lớn nhất:
                                   ShipToGroupID MainIsland  DemandPallet    Ton  SoShipTo  SoSP
          ID_JV_Jawa Barat_Bekasi_Cikarang Utara       Java       1251.21 334.63       145  1576
                                      118438-598       Java       1218.47 332.22         1   260
       ID_JV_Jawa Timur_Kota Surabaya_Sambikerep       Java       1002.67 249.94        33  1481
                    ID_JV_Banten_Tangerang_Curug       Java        875.82 224.26        55  1555
ID_JV_Dki Jakarta_Kota Jakarta Utara_Penjaringan       Java        827.20 206.89        45  1432

5 nhóm nhỏ nhất:
                                 ShipToGroupID MainIsland  DemandPallet    Ton  SoShipTo  SoSP
          ID_JV_Jawa Tengah_Purworejo_Kutoarjo       Java        0.0004 0.0001         1     3
ID_SMT_Sumatera Utara_Tapanuli Utara_Sipahutar    Sumatra        0.0010 0.0004         1     3
          ID_SMT_Sumatera Utara_Langkat_Binjai    Sumatra        0.0014 0.0004         1     2
   

In [8]:
# ===== CẦU THEO KÊNH BÁN: CÒN KHÁCH "Manufacturer" NÀO KHÁC KHÔNG? =====
kenh = cmf[["LocationID", "Channel", "ChannelName"]].astype(str)
x = dm.merge(kenh, left_on="ShipToID", right_on="LocationID", how="left")
bang = x.groupby(["Channel", "ChannelName"]).agg(so_khach=("ShipToID", "nunique"),
                                                 pallet=("Pallet", "sum"), tan=("Tan", "sum"))
bang["ty_le_pallet"] = bang["pallet"] / bang["pallet"].sum()
print(bang.sort_values("pallet", ascending=False).round(3).to_string())


                                                                 so_khach     pallet       tan  ty_le_pallet
Channel                         ChannelName                                                                 
Modern Trade                    National Minimarket                   174  15031.944  3895.388         0.207
Health & Beauty                 Cosmetics Store Retail               8089  11810.006  3019.850         0.163
E-Commerce                      Local E-Commerce                      478  10624.175  2615.670         0.146
Modern Trade                    Local Hyper/Super                    1476   6909.436  1899.824         0.095
Health & Beauty                 Cosmetic Store Wholesaler             396   6157.905  1575.877         0.085
                                Cosmetic Store Lifestyle              919   5901.504  1482.217         0.081
                                HABA                                  856   4162.877  1033.415         0.057
General Trade      

### 1.3.3 Export

Lưu `Model Outputs/demand.parquet` → `Multi-Objective Optimization.ipynb` đọc.

In [9]:
# ===== 1.3.3 XUẤT BẢNG CẦU =====
# Kiểm tra lần cuối trước khi lưu
assert demand["ShipToGroupID"].is_unique                               # mỗi nhóm 1 dòng
assert demand.notna().all().all()                                      # không ô trống
assert (demand["DemandPallet"] > 0).all() and (demand["Ton"] > 0).all()       # cầu dương

out_file = OUT / "demand.parquet"
demand.to_parquet(out_file, index=False)                               # index=False: không lưu cột số thứ tự
check = pd.read_parquet(out_file)                                      # đọc lại để chắc file ghi đúng
assert check.shape == demand.shape
print(f"Đã lưu: {out_file} | {len(check):,} nhóm | Σ DemandPallet = {check['DemandPallet'].sum():,.0f} pallet/năm "
      f"| Σ tấn = {check['Ton'].sum():,.0f}")

Đã lưu: ..\Model Outputs\demand.parquet | 3,346 nhóm | Σ DemandPallet = 72,624 pallet/năm | Σ tấn = 18,721


## 1.4 Empirical evidence of demand variability (industry characteristics)

### 1.4.1 Seasonality

*(code: chỉ số mùa vụ theo tháng)*

### 1.4.2 Group-level volatility

*(code: CV theo nhóm)*

### 1.4.3 Non-simultaneous peaks

*(code: tổng đỉnh từng nhóm so với đỉnh toàn quốc → đỉnh toàn quốc chỉ +20% so với trung bình, dùng làm mức cho kịch bản độ nhạy)*

### 1.4.4 Demand concentration and fragmentation

*(code: tập trung theo nhóm, số ship-to, kích thước dòng đơn)*

### 1.4.5 Product characteristics

*(code: SKU đuôi dài, kg/pallet)*

## 1.5 Implications for the model

*(nội dung: bảng đặc tính quan sát được → thành phần mô hình; cầu d_k cho notebook 2; mức cầu cho kịch bản độ nhạy +10%, +20%)*